---
date: "2026-06-17"
date-modified: last-modified
format:
  html:
    toc: true
---

# Problem Set: Random Variables

## Problem 1: Stat 110 Strategic Practice 3, Fall 2011, Blitzstein. P. 1

**Problem Statement:**
Consider the Monty Hall problem, except that Monty enjoys opening Door 2 more than he enjoys opening Door 3, and if he has a choice between opening these two doors, he opens Door 2 with probability $p$, where $1/2 \le p \le 1$.

To recap: suppose you are forced to select door 1.

(a) Find the unconditional probability that the strategy of always switching succeeds.
(b) Find the probability that the strategy of always switching succeeds, given that Monty opens Door 2.
(c) Find the probability that the strategy of always switching succeeds, given that Monty opens Door 3.

**Mathematical Solution:**

Let $D_i$ be the event that the car is behind Door $i$, for $i \in \{1, 2, 3\}$. Initially, $P(D_i) = 1/3$.
Let $M_j$ be the event that Monty opens Door $j$, for $j \in \{2, 3\}$.

Based on the rules of the game (and given we have selected Door 1), we can establish Monty's conditional probabilities for opening doors:

* If the car is behind Door 1 ($D_1$), Monty has a choice. By the prompt's definition: $P(M_2|D_1) = p$ and $P(M_3|D_1) = 1 - p$.
* If the car is behind Door 2 ($D_2$), Monty *must* open Door 3 to avoid revealing the car: $P(M_2|D_2) = 0$ and $P(M_3|D_2) = 1$.
* If the car is behind Door 3 ($D_3$), Monty *must* open Door 2: $P(M_2|D_3) = 1$ and $P(M_3|D_3) = 0$.

### Part (a): Unconditional Probability
We want to find the overall probability of winning by switching, denoted as $P(S)$. 
The switching strategy wins if and only if the car is *not* behind our initial choice (Door 1). 
$$P(S) = P(D_2 \cup D_3) = 1 - P(D_1) = 1 - \frac{1}{3} = \frac{2}{3}$$
Monty's specific bias $p$ does not matter unconditionally because we are not restricting the sample space to a specific door being opened.

### Part (b): Conditional Probability given Monty opens Door 2
We want to find the probability that switching succeeds given $M_2$. Since we chose Door 1 and Monty opened Door 2, switching means choosing Door 3. Therefore, winning by switching is exactly the event $D_3$. We need to find $P(D_3|M_2)$.

By [Bayes' Theorem](conditional-probability.ipynb#bayes-theorem):
$$P(D_3|M_2) = \frac{P(M_2|D_3)P(D_3)}{P(M_2)}$$

We use the [LoTP](law-of-total-probability.ipynb#theorem-statement) to expand the denominator $P(M_2)$:
$$P(M_2) = P(M_2|D_1)P(D_1) + P(M_2|D_2)P(D_2) + P(M_2|D_3)P(D_3)$$
$$P(M_2) = \left(p \cdot \frac{1}{3}\right) + \left(0 \cdot \frac{1}{3}\right) + \left(1 \cdot \frac{1}{3}\right) = \frac{p + 1}{3}$$

Now, substitute this back into Bayes' Theorem:
$$P(D_3|M_2) = \frac{1 \cdot (1/3)}{(p + 1)/3} = \frac{1}{p + 1}$$

*(Sanity Check: If Monty is perfectly unbiased ($p=1/2$), this formula yields $1/(1.5) = 2/3$, matching the standard Monty Hall result.)*

### Part (c): Conditional Probability given Monty opens Door 3
Similarly, we want to find the probability that switching succeeds given $M_3$. Since we chose Door 1 and Monty opened Door 3, switching means choosing Door 2. We need to find $P(D_2|M_3)$.
By Bayes' Theorem:
$$P(D_2|M_3) = \frac{P(M_3|D_2)P(D_2)}{P(M_3)}$$

We expand the denominator $P(M_3)$:
$$P(M_3) = P(M_3|D_1)P(D_1) + P(M_3|D_2)P(D_2) + P(M_3|D_3)P(D_3)$$
$$P(M_3) = \left((1-p) \cdot \frac{1}{3}\right) + \left(1 \cdot \frac{1}{3}\right) + \left(0 \cdot \frac{1}{3}\right) = \frac{1 - p + 1}{3} = \frac{2 - p}{3}$$

Substituting this back into Bayes' Theorem:
$$P(D_2|M_3) = \frac{1 \cdot (1/3)}{(2 - p)/3} = \frac{1}{2 - p}$$

In [1]:
import random

def simulate_biased_monty_hall(trials=100000, p=0.75):
    # Counters for our scenarios
    switch_wins_total = 0
    
    monty_opened_2_count = 0
    switch_wins_given_m2 = 0
    
    monty_opened_3_count = 0
    switch_wins_given_m3 = 0

    for _ in range(trials):
        # 1. Place the car randomly (Doors 1, 2, or 3)
        car_door = random.choice([1, 2, 3])
        
        # 2. Player always chooses Door 1 (as per the prompt)
        player_door = 1
        
        # 3. Monty makes his choice
        if car_door == 1:
            # Monty has a choice. He chooses Door 2 with probability p.
            if random.random() < p:
                monty_door = 2
            else:
                monty_door = 3
        elif car_door == 2:
            # Monty MUST open Door 3
            monty_door = 3
        else: # car_door == 3
            # Monty MUST open Door 2
            monty_door = 2
            
        # 4. Evaluate switching
        # Since player picked 1, and Monty picked 'monty_door', the switch door is the remaining one
        switch_door = [d for d in [1, 2, 3] if d != player_door and d != monty_door][0]
        
        # Did switching win?
        won_by_switching = (switch_door == car_door)
        
        # 5. Update Total Counters
        if won_by_switching:
            switch_wins_total += 1
            
        # 6. Update Conditional Counters
        if monty_door == 2:
            monty_opened_2_count += 1
            if won_by_switching:
                switch_wins_given_m2 += 1
        elif monty_door == 3:
            monty_opened_3_count += 1
            if won_by_switching:
                switch_wins_given_m3 += 1

    # Calculate probabilities
    p_win_total = switch_wins_total / trials
    p_win_m2 = switch_wins_given_m2 / monty_opened_2_count if monty_opened_2_count > 0 else 0
    p_win_m3 = switch_wins_given_m3 / monty_opened_3_count if monty_opened_3_count > 0 else 0
    
    print(f"--- Simulation Results (Bias p={p}, Trials={trials}) ---")
    print(f"(a) Unconditional P(Win by Switching): {p_win_total:.4f}")
    print(f"(b) P(Win by Switching | Monty opens Door 2): {p_win_m2:.4f}")
    print(f"(c) P(Win by Switching | Monty opens Door 3): {p_win_m3:.4f}")

# Run the simulation with p = 0.75 (3/4 bias toward Door 2)
simulate_biased_monty_hall(trials=100000, p=0.75)

--- Simulation Results (Bias p=0.75, Trials=100000) ---
(a) Unconditional P(Win by Switching): 0.6671
(b) P(Win by Switching | Monty opens Door 2): 0.5707
(c) P(Win by Switching | Monty opens Door 3): 0.8020



## Problem 2: Stat 110 Strategic Practice 3, Fall 2011, Blitzstein. P.2

**Problem Statement:**

For each statement below, either show that it is true or give a counterexample. Throughout, $X, Y, Z$ are discrete random variables.

(a) If $X$ and $Y$ are independent and $Y$ and $Z$ are independent, then $X$ and $Z$ are independent.
(b) If $X$ and $Y$ are independent, then they are conditionally independent given $Z$.
(c) If $X$ and $Y$ are conditionally independent given $Z$, then they are independent.
(d) If $X$ and $Y$ have the same distribution given $Z$, i.e., for all $a$ and $z$, we have $P(X = a | Z = z) = P(Y = a | Z = z)$, then $X$ and $Y$ have the same distribution.

---

**Mathematical Solution:**

**(a) False.** [Independence](independence-of-rvs.ipynb) is not a transitive property. 

**Counterexample:** Let $X$ be the result of a fair coin flip. Let $Z = X$ (meaning $Z$ is literally the exact same coin flip). Now, let $Y$ be the result of a completely separate, second coin flip. 
Because $Y$ is a separate flip, $X$ and $Y$ are independent. For the same reason, $Y$ and $Z$ are independent. However, $X$ and $Z$ are identical—meaning if you know $X$, you know $Z$ with 100% certainty. They are perfectly dependent.

**(b) False.** This is a classic statistical trap often related to "explaining away" or Berkson's Paradox. Two variables can be completely independent in a vacuum, but become highly dependent once you observe a shared outcome (a common effect).

**Counterexample:** Let $X$ and $Y$ be two independent fair coin flips (where $1$ is Heads, $0$ is Tails). Let $Z = X + Y$ (the total number of Heads rolled). 
Unconditionally, $X$ and $Y$ are independent. However, suppose we condition on $Z = 1$ (we know exactly one Head was rolled). If I now tell you that $X = 1$, you immediately know with absolute certainty that $Y = 0$. Given the knowledge of $Z$, $X$ and $Y$ become perfectly dependent.

**(c) False.**
This is the inverse of the trap from part (b). Variables can be conditionally independent but unconditionally dependent. This typically happens when two variables share a common, unobserved cause.

**Counterexample:** Suppose $Z$ represents the unknown bias of a coin you just pulled from your pocket (maybe it's a fair coin, maybe it's a trick double-headed coin). Let $X$ and $Y$ be two consecutive flips of this chosen coin. 
If you *know* the coin's exact bias (conditioning on $Z$), the flips $X$ and $Y$ are independent Bernoulli trials. However, if you *don't* know $Z$, seeing $X$ land Heads provides evidence that you might be holding the trick double-headed coin. This new belief suddenly makes it much more likely that $Y$ will also land Heads. Because observing $X$ changes your expectation of $Y$, they are unconditionally dependent.

**(d) True.**
This is a direct, elegant application of the **Law of Total Probability**. 
To find the unconditional distribution of $X$, we sum over all possible conditions of $Z$:
$$P(X = a) = \sum_z P(X = a \mid Z = z) P(Z = z)$$

Because we are explicitly given that the conditional distributions are identical ($P(X = a \mid Z = z) = P(Y = a \mid Z = z)$), we can seamlessly substitute $Y$ into the equation:
$$P(X = a) = \sum_z P(Y = a \mid Z = z) P(Z = z)$$

Applying the Law of Total Probability in reverse to the right side of the equation yields the unconditional distribution of $Y$:
$$P(X = a) = P(Y = a)$$

Since this holds for all values of $a$, $X$ and $Y$ must have the exact same unconditional distribution. $\blacksquare$

> **References & Acknowledgments:**
> 
> Problems adapted from *Introduction to Probability* by Joseph K. Blitzstein and Jessica Hwang.
>
> Problems adapted from *Stat 110 Strategic Practice Problems*, by Prof. Joe Blitzstein (Harvard University, [stat110.hsites.harvard.edu](https://stat110.hsites.harvard.edu/)).